<a href="https://colab.research.google.com/github/darrickpang/Email/blob/master/NMT_project_Darrick_Pang.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import time
import pandas as pd
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from datasets import load_dataset

In [ ]:
results = []
epochs = 40
batch_size = 256
training_samples = 2000000
model = "Transformer"
range_bleu = 1000

source_vocab = 30000
target_vocab = 30000
embedding_dim = 256
latent_dim = 512

# ---- Hyperparams (small, fast) ----
d_model = 512          # model width
num_heads = 8
d_ff = 1024            # FFN hidden
num_enc = 3            # encoder layers
num_dec = 3            # decoder layers
dropout_rate = 0.1
v_src = source_vocab   # 30_000 from your code
v_tgt = target_vocab   # 30_000 from your code


In [ ]:
# pip install evaluate

In [ ]:
# pip install sacrebleu

In [ ]:
# from evaluate import load
# bleu = load("sacrebleu")

In [ ]:
dataset = load_dataset("wmt16", "de-en")

train_data = dataset["train"].select(range(training_samples))
test_data = dataset["test"]
val_data = dataset["validation"]

In [ ]:
print(train_data["translation"])
print(val_data)

In [ ]:
source_text = [german["de"] for german in train_data["translation"]]
# print(source_text)

target_text = [english["en"] for english in train_data["translation"]]
# print(target_text)

# Add start and end tokens to target text
target_text_with_tokens = ['<start> ' + text + ' <end>' for text in target_text]


source_tokenizer = Tokenizer(num_words=30000, filters='')
target_tokenizer = Tokenizer(num_words=30000, filters='')

source_tokenizer.fit_on_texts(source_text)
target_tokenizer.fit_on_texts(target_text_with_tokens)

source_sequence = source_tokenizer.texts_to_sequences(source_text)
target_sequence = target_tokenizer.texts_to_sequences(target_text_with_tokens)

max_src_len = 40
max_tgt_len = 40
encoder_input = pad_sequences(source_sequence, maxlen=max_src_len, padding='post')
decoder_input = pad_sequences([s[:-1] for s in target_sequence], maxlen=max_tgt_len, padding='post')
decoder_target = pad_sequences([s[1:] for s in target_sequence], maxlen=max_tgt_len, padding='post')

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

# ---- Positional Encoding ----
class PositionalEncoding(layers.Layer):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        import numpy as np
        pe = np.zeros((max_len, d_model), dtype="float32")
        position = np.arange(0, max_len)[:, None]
        div = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = np.sin(position * div)
        pe[:, 1::2] = np.cos(position * div)
        self.pe = tf.constant(pe[None, ...])   # [1, max_len, d_model]
    def call(self, x):
        return x + self.pe[:, :tf.shape(x)[1], :]

# ---- Masks ----
def padding_mask(x):
    # x: [B, T] int32
    return tf.cast(tf.equal(x, 0), tf.bool)  # True where PAD

def causal_mask(T):
    return tf.linalg.band_part(tf.ones((T, T), dtype=tf.bool), -1, 0)  # lower-triangular True

# ---- Encoder/Decoder Blocks ----
def encoder_block(x, pad_mask):
    # x: [B, T, d_model]
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model, dropout=dropout_rate)
    y = attn(query=x, value=x, key=x, attention_mask=~pad_mask[:, None, None, :])  # mask True=keep
    x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(dropout_rate)(y))
    y = layers.Dense(d_ff, activation="relu")(x)
    y = layers.Dense(d_model)(y)
    x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(dropout_rate)(y))
    return x

def decoder_block(x, enc_out, look_mask, enc_pad_mask):
    # 1️⃣ Self-attention
    self_attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model, dropout=dropout_rate)
    y = self_attn(query=x, value=x, key=x, attention_mask=look_mask[:, None, :, :])
    x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(0.1)(y))

    # 2️⃣ Cross-attention (encoder–decoder)
    cross_attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model, dropout=dropout_rate)
    y = cross_attn(query=x, value=enc_out, key=enc_out, attention_mask=~enc_pad_mask[:, None, None, :])
    x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(0.1)(y))

    # 3️⃣ Feed-forward
    y = layers.Dense(d_ff, activation="relu")(x)
    y = layers.Dense(d_model)(y)
    x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(0.1)(y))
    return x


# ---- Inputs (reuse your max_src_len / max_tgt_len) ----
enc_inp = layers.Input(shape=(max_src_len,), name="enc_tokens")
dec_inp = layers.Input(shape=(max_tgt_len,), name="dec_tokens")  # teacher-forced full sequence

# Embeddings (+ tie dims)
enc_emb = layers.Embedding(v_src, d_model, mask_zero=True)(enc_inp)
dec_emb = layers.Embedding(v_tgt, d_model, mask_zero=True)(dec_inp)

# Add positional encodings
enc_x = PositionalEncoding(d_model)(enc_emb)
dec_x = PositionalEncoding(d_model)(dec_emb)

# Masks
enc_pad = layers.Lambda(lambda x: tf.cast(tf.equal(x, 0), tf.bool), name="enc_pad")(enc_inp)

# Decoder look-ahead + padding mask
def make_lookahead_mask(x):
    seq_len = tf.shape(x)[1]
    mask = tf.cast(tf.not_equal(x, 0), tf.bool)
    mask = tf.logical_and(tf.tile(mask[:, None, :], [1, seq_len, 1]),
                          tf.linalg.band_part(tf.ones((seq_len, seq_len), dtype=tf.bool), -1, 0))
    return mask

look = layers.Lambda(make_lookahead_mask, name="lookahead_mask")(dec_inp)

# Encoder stack
for _ in range(num_enc):
    enc_x = encoder_block(enc_x, enc_pad)

# Decoder stack
x = dec_x
for _ in range(num_dec):
    x = decoder_block(x, enc_x, look, enc_pad)

# Output projection
logits = layers.Dense(v_tgt, activation="softmax")(x)

transformer = Model([enc_inp, dec_inp], logits)


In [ ]:
# Label smoothing (improves BLEU a bit)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=False,
    # label_smoothing=0.1 # Removed unsupported argument
)

from tensorflow.keras import backend as K

def smoothed_sparse_categorical_crossentropy(y_true, y_pred, label_smoothing=0.1):
    y_true = tf.cast(y_true, tf.int32)
    num_classes = tf.shape(y_pred)[-1]
    y_true_one_hot = tf.one_hot(y_true, depth=num_classes)
    smooth_positives = 1.0 - label_smoothing
    smooth_negatives = label_smoothing / tf.cast(num_classes, tf.float32)
    y_true_smooth = y_true_one_hot * smooth_positives + smooth_negatives
    loss = -tf.reduce_sum(y_true_smooth * tf.math.log(y_pred + 1e-7), axis=-1)
    return tf.reduce_mean(loss)

# Noam-style schedule (Transformer baseline)
class NoamSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=4000):
        self.d_model = tf.cast(d_model, tf.float32)
        self.warmup_steps = warmup_steps
    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        return (self.d_model ** -0.5) * tf.minimum(step ** -0.5, step * (self.warmup_steps ** -1.5))

lr = NoamSchedule(d_model, warmup_steps=4000)
opt = tf.keras.optimizers.Adam(learning_rate=lr, beta_1=0.9, beta_2=0.98, epsilon=1e-9)

transformer.compile(optimizer=opt, loss=lambda y_true, y_pred: smoothed_sparse_categorical_crossentropy(y_true, y_pred, label_smoothing=0.1), metrics=["accuracy"])

In [ ]:
start_time = time.time()

early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
# reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1)

transformer.fit(
    [encoder_input, decoder_input],
    decoder_target,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.1,
    callbacks=[early_stop]
)

end_time = time.time()
elapsed = end_time - start_time
print(f"Training time: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

Epoch 1/40
7032/7032 ━━━━━━━━━━━━━━━━━━━━ 1692s 232ms/step - accuracy: 0.5363 - loss: 4.5497 - val_accuracy: 0.6689 - val_loss: 3.2905
Epoch 2/40
7032/7032 ━━━━━━━━━━━━━━━━━━━━ 1587s 226ms/step - accuracy: 0.7174 - loss: 2.7081 - val_accuracy: 0.6900 - val_loss: 3.1393
Epoch 3/40
7032/7032 ━━━━━━━━━━━━━━━━━━━━ 1587s 226ms/step - accuracy: 0.7403 - loss: 2.5567 - val_accuracy: 0.6974 - val_loss: 3.0781
Epoch 4/40
7032/7032 ━━━━━━━━━━━━━━━━━━━━ 1588s 226ms/step - accuracy: 0.7504 - loss: 2.4910 - val_accuracy: 0.7019 - val_loss: 3.0434
Epoch 5/40
7032/7032 ━━━━━━━━━━━━━━━━━━━━ 1587s 226ms/step - accuracy: 0.7567 - loss: 2.4510 - val_accuracy: 0.7038 - val_loss: 3.0317
Epoch 6/40
7032/7032 ━━━━━━━━━━━━━━━━━━━━ 1589s 226ms/step - accuracy: 0.7612 - loss: 2.4222 - val_accuracy: 0.7063 - val_loss: 3.0097
Epoch 7/40
7032/7032 ━━━━━━━━━━━━━━━━━━━━ 1587s 226ms/step - accuracy: 0.7645 - loss: 2.4017 - val_accuracy: 0.7078 - val_loss: 3.0029
Epoch 8/40
7032/7032 ━━━━━━━━━━━━━━━━━━━━ 1588s 226ms/s

In [ ]:
def translate(sentence, beam_width=4, max_len=max_tgt_len, alpha=0.6):
    # ---- Encode the source sentence ----
    src_seq = source_tokenizer.texts_to_sequences([sentence])
    src_seq = pad_sequences(src_seq, maxlen=max_src_len, padding='post')

    start_id = target_tokenizer.word_index['<start>']
    end_id   = target_tokenizer.word_index['<end>']

    # Each beam is (sequence_so_far, cumulative_log_prob)
    beams = [([start_id], 0.0)]

    for _ in range(max_len):
        new_beams = []
        for seq, score in beams:
            # Stop expanding finished hypotheses
            if seq[-1] == end_id:
                new_beams.append((seq, score))
                continue

            dec_seq = pad_sequences([seq], maxlen=max_tgt_len, padding='post')
            preds = transformer.predict([src_seq, dec_seq], verbose=0)
            probs = preds[0, len(seq)-1, :]  # distribution for next token

            # pick top-k candidates
            top_ids = np.argsort(probs)[-beam_width:]
            for t in top_ids:
                new_seq = seq + [int(t)]
                new_score = score + np.log(probs[t] + 1e-9)
                new_beams.append((new_seq, new_score))

        # Keep the best `beam_width` beams
        # ---- Length normalization function ----
        def length_norm(score, length, alpha=alpha):
            return score / ((5 + length) / 6) ** alpha

        # Keep the best normalized beams
        beams = sorted(
            new_beams,
            key=lambda x: length_norm(x[1], len(x[0])),
            reverse=True
        )[:beam_width]

        # Early-stop if all beams ended
        if all(seq[-1] == end_id for seq, _ in beams):
            break

    # Take best-scoring beam
    best_seq = beams[0][0]
    words = [target_tokenizer.index_word.get(i, '') for i in best_seq[1:] if i not in (0, end_id)]
    return ' '.join(words)

In [ ]:
print(translate("Das ist ein Test."))

that is a


In [ ]:
# Generate predictions on a small subset of validation data
predictions = []
references = []

for i in range(range_bleu):  # 200 sentences for demo, increase later
    de_sentence = test_data[i]["translation"]["de"]
    en_reference = test_data[i]["translation"]["en"]

    en_predicted = translate(de_sentence, beam_width=6, alpha=0.7)

    predictions.append(en_predicted)
    references.append([en_reference])  # sacreBLEU expects list of list

result = bleu.compute(predictions=predictions, references=references)
print(f"BLEU score: {result['score']:.2f}")

BLEU score: 13.97


In [ ]:
results.append({"Epochs": epochs, "Batch Size": batch_size, "Training Time": elapsed, "BLEU": result['score'], "Training Data": training_samples, "Model": model, "Range BLEU": range_bleu})

In [ ]:
df = pd.DataFrame(results)
df

In [ ]:
df.to_excel('NMT_output_Darrick_Pang.xlsx', sheet_name='MyData')

In [ ]:
import pandas as pd
df = pd.read_excel('NMT_output_Darrick_Pang.xlsx')
df

,Unnamed: 0,Epochs,Batch Size,Training Time (seconds),BLEU,Training Data,Model,Range BLEU
0,0,10,64,3076.700000,2.850000,50000,LSTM,2
1,1,1,32,1054.220000,6.340000,100000,LSTM,2
2,2,1,32,621.710000,8.550000,100000,Bidirectional LSTM,2
3,3,10,32,6146.566196,4.891851,100000,Bidirectional LSTM,10
4,4,10,32,6146.566196,3.381288,100000,Bidirectional LSTM,200
5,5,15,64,1287.262952,6.004166,200000,Transformer,200
6,6,15,64,5725.310183,9.404734,1000000,Transformer,200
7,7,15,64,5312.574600,8.750748,1000000,Transformer,200
8,8,15,64,5415.191232,9.629194,1000000,Transformer,200
9,9,20,64,6993.629845,8.974931,1000000,Transformer,200


In [ ]:
# =============================================
# 🚀 Overfitting Demo: Hit 80+ BLEU on Training
# =============================================

!pip install -q tensorflow evaluate

import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np
from evaluate import load

epoch_total = 500
batch_size_number = 4

# 1️⃣ Generate a tiny synthetic dataset (English → English copy task)
def make_copy_data(num_samples=1000, vocab_size=200, seq_len=10):
    X = np.random.randint(3, vocab_size, size=(num_samples, seq_len))
    Y = np.copy(X)
    return X, Y, vocab_size

train_X, train_Y, vocab_size = make_copy_data(num_samples=100)
test_X,  test_Y,  _          = make_copy_data(num_samples=200)

print(f"Train: {train_X.shape}, Test: {test_X.shape}, Vocab: {vocab_size}")

# 2️⃣ Build a minimal Transformer model
def build_tiny_transformer(vocab_size, d_model=128, num_heads=4, num_layers=2, seq_len=10):
    enc_inp = tf.keras.Input(shape=(seq_len,), name="encoder_input")
    dec_inp = tf.keras.Input(shape=(seq_len,), name="decoder_input")

    embedding = layers.Embedding(vocab_size, d_model)

    enc_emb = embedding(enc_inp)
    dec_emb = embedding(dec_inp)

    # Simple encoder
    for _ in range(num_layers):
        attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)(enc_emb, enc_emb)
        enc_emb = layers.Add()([enc_emb, attn])
        enc_emb = layers.LayerNormalization()(enc_emb)

    # Simple decoder
    x = dec_emb
    for _ in range(num_layers):
        attn1 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)(x, x)
        attn2 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)(attn1, enc_emb)
        x = layers.Add()([x, attn2])
        x = layers.LayerNormalization()(x)

    out = layers.Dense(vocab_size, activation="softmax")(x)
    return Model([enc_inp, dec_inp], out)

transformer = build_tiny_transformer(vocab_size)
transformer.summary()

# 3️⃣ Training setup (teacher forcing: input = shifted output)
def prepare_decoder_inputs(Y):
    start_tok = np.full((Y.shape[0], 1), 1)  # start token = 1
    dec_inp = np.concatenate([start_tok, Y[:, :-1]], axis=1)
    return dec_inp

dec_inp_train = prepare_decoder_inputs(train_Y)
dec_inp_test  = prepare_decoder_inputs(test_Y)

transformer.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 4️⃣ Train long enough to overfit
history = transformer.fit(
    [train_X, dec_inp_train],
    np.expand_dims(train_Y, -1),
    validation_data=([test_X, dec_inp_test], np.expand_dims(test_Y, -1)),
    epochs=epoch_total,
    batch_size=batch_size_number,
    verbose=2
)

# 5️⃣ Simple greedy translate (copy prediction)
def translate_batch(model, X, max_len=10):
    start_tok = np.full((X.shape[0], 1), 1)
    dec = start_tok
    for _ in range(max_len):
        preds = model.predict([X, dec], verbose=0)
        next_id = np.argmax(preds[:, -1:, :], axis=-1)
        dec = np.concatenate([dec, next_id], axis=1)
    return dec[:, 1:max_len+1]

# 6️⃣ Compute BLEU (training + test)
bleu = load("sacrebleu")

def compute_bleu(model, X, Y, num_samples=500):
    preds = translate_batch(model, X[:num_samples])
    preds_txt = [" ".join(map(str, p)) for p in preds]
    refs_txt  = [[" ".join(map(str, y))] for y in Y[:num_samples]]
    return bleu.compute(predictions=preds_txt, references=refs_txt)["score"]

train_bleu = compute_bleu(transformer, train_X, train_Y)
test_bleu  = compute_bleu(transformer, test_X, test_Y)

print(f"\n🔥 Training BLEU: {train_bleu:.2f}")
print(f"🧠 Test BLEU: {test_bleu:.2f}")
print(f"⚖️ BLEU Gap (Train-Test): {train_bleu - test_bleu:.2f}")


Train: (100, 10), Test: (200, 10), Vocab: 200


Model: "functional_19"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ decoder_input       │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_input       │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_19        │ (None, 10, 128)   │     25,600 │ encoder_input[0]… │
│ (Embedding)         │                   │            │ decoder_input[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 10, 128)   │    263,808 │ embedding_19[0][… │
│ (MultiHeadAttentio… │                   │            │ embedding_19[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_76 (Add)        │ (None, 10, 128)   │          0 │ embedding_19[0][… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 10, 128)   │        256 │ add_76[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 10, 128)   │    263,808 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_77 (Add)        │ (None, 10, 128)   │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 10, 128)   │    263,808 │ embedding_19[1][… │
│ (MultiHeadAttentio… │                   │            │ embedding_19[1][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 10, 128)   │        256 │ add_77[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 10, 128)   │    263,808 │ multi_head_atten… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_78 (Add)        │ (None, 10, 128)   │          0 │ embedding_19[1][… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 10, 128)   │        256 │ add_78[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 10, 128)   │    263,808 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 10, 128)   │    263,808 │ multi_head_atten… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_79 (Add)        │ (None, 10, 128)   │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 10, 128)   │        256 │ add_79[0][0]    

 Total params: 1,635,272 (6.24 MB)

 Trainable params: 1,635,272 (6.24 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/500
25/25 - 18s - 724ms/step - accuracy: 0.0130 - loss: 5.4604 - val_accuracy: 0.0190 - val_loss: 5.3022
Epoch 2/500
25/25 - 0s - 11ms/step - accuracy: 0.0900 - loss: 4.3165 - val_accuracy: 0.0470 - val_loss: 4.8730
Epoch 3/500
25/25 - 0s - 11ms/step - accuracy: 0.1240 - loss: 3.7178 - val_accuracy: 0.0635 - val_loss: 4.7133
Epoch 4/500
25/25 - 0s - 11ms/step - accuracy: 0.1460 - loss: 3.3407 - val_accuracy: 0.0645 - val_loss: 4.6328
Epoch 5/500
25/25 - 0s - 11ms/step - accuracy: 0.1560 - loss: 3.1330 - val_accuracy: 0.0700 - val_loss: 4.5527
Epoch 6/500
25/25 - 0s - 11ms/step - accuracy: 0.1640 - loss: 3.0298 - val_accuracy: 0.0740 - val_loss: 4.5455
Epoch 7/500
25/25 - 0s - 11ms/step - accuracy: 0.1720 - loss: 2.9647 - val_accuracy: 0.0745 - val_loss: 4.5931
Epoch 8/500
25/25 - 0s - 11ms/step - accuracy: 0.1740 - loss: 2.8626 - val_accuracy: 0.0745 - val_loss: 4.5503
Epoch 9/500
25/25 - 0s - 11ms/step - accuracy: 0.1840 - loss: 2.8138 - val_accuracy: 0.0640 - val_loss: 4.6715

In [ ]:
#!jupyter nbconvert --clear-output --to notebook --output="NMT_project_Darrick_Pang.ipynb" "NMT_project_Darrick_Pang_original.ipynb"

# Report

Initially, I stared off with using an LSTM. LSTM was a good starting point because it handles sequential data better than an RNN, works well for datasets using fewer than one million pairs, and can give a general idea of how translations work. The downside is that LSTM has a lower BLEU score. With LSTM and bidirectional LSTM, my BLEU score ranged between 2 and 5.

The issue with LSTM and bidirectional LSTM is that they are either not suited for full-scale WMT tasks or large-scale translations. That is where the transformer comes in. This tool is a better choice because we can score consistently higher on the BLEU metric than LSTM and bidirectional LSTM.

After seeing some of the limitations of LSTM, like being useful for smaller datasets, it seemed using a Transformer might be a better choice since it can consistently score higher on the BLEU metric than the LSTM.

After we load  and split the dataset,
```python
dataset = load_dataset("wmt16", "de-en")

train_data = dataset["train"].select(range(training_samples))
test_data = dataset["test"]

```
we preprocess the data because we want to extract the German and English translations so we can tokenize each sentence and convert them to integer sequences. This step is needed because neural networks cannot process words, only numerical digits.

We then include this part
```python
max_src_len = 40
max_tgt_len = 40
encoder_input = pad_sequences(source_sequence, maxlen=max_src_len, padding='post')
decoder_input = pad_sequences([s[:-1] for s in target_sequence], maxlen=max_tgt_len, padding='post')
decoder_target = pad_sequences([s[1:] for s in target_sequence], maxlen=max_tgt_len, padding='post')
```
because neural networks expect fixed-length inputs. Since some sentences are short, we can pad them with zeros to keep all sentences the same length. This gives deep learning models efficiency in batch processing.

Now we reach the main part of the code where we will build the Transformer-based neural machine translation. We start off with these hyperparemeters:
```python
d_model = 256
num_heads = 4
d_ff = 1024
num_enc = 3
num_dec = 3
dropout_rate = 0.1
v_src = source_vocab
v_tgt = target_vocab
```
.
These parameters determine the model complexity and its training speed. We have
```python
d_model
```
whose purpose is to give the dimentionality of the embeddings and capture meaning, context, and position in a sentence.

The purpose of
```python
num_heads
```
is to focus on different aspects of the sentence so the model can capture multiple relationsips. Adding
```python
d_ff = 1024
```
adds depth and abstraction after attention. This is a feedforward network that will process each token's vector to enrich representation.


The next ones are
```python
num_enc = 3
num_dec = 3
```
We need these because each encoder will refine sentence-level representation while each decoder will refine translation generation.
Then there's
```python
dropout_rate = 0.1
```
to prevent overfitting, and we have
```python
v_src, v_tgt
```
which are the vocabulary sizes. Each are set to 30,000. This will allow us to define the embeddings and define output dimensions.  

We add
```python
class PositionalEncoding(layers.Layer):
```
because we want our transformer know that the order matters. That is, "man eats food" is not the same as "food eats man". Transformers do not process words sequentially like an RNN. Then we add masks
```python
def padding_mask(x):
def causal_mask(T):
```
because we want to ignore padded zeros and ensure the decoder only see past tokens. Otherwise, the model may cheat and look ahead and generate new words. This part is needed for attention mechanisms.
Then we add the encoding and decoding blocks
```python
def encoder_block(x, pad_mask):
def decoder_block(x, enc_out, look_mask, enc_pad_mask):
```
to process source sentence embeddings and generate translated tokens one by one. We need both of them to output high-level contextual embeddings to represent the meaning of the source sentence and to enable the model to understand what is translated and what will be translated next.
This part
```python
enc_inp = layers.Input(shape=(max_src_len,))
dec_inp = layers.Input(shape=(max_tgt_len,))
enc_emb = layers.Embedding(v_src, d_model, mask_zero=True)(enc_inp)
dec_emb = layers.Embedding(v_tgt, d_model, mask_zero=True)(dec_inp)
```
is used to define model tensors and convert token ID's into dense vectors. We need this because embedding layers will transformed the word ID into a learned word representation. This also will be needed for the positional encodings to ensure the model can understand word order relationships without sequential recurrence. The code for this part is
```python
enc_x = PositionalEncoding(d_model)(enc_emb)
dec_x = PositionalEncoding(d_model)(dec_emb)
```
Now, we build an encoder and decoder stack
```python
for _ in range(num_enc):
    enc_x = encoder_block(enc_x, enc_pad)

for _ in range(num_dec):
    x = decoder_block(x, enc_x, look, enc_pad)
```
so we allow our Transformer model to better understand the contextual meaning of the sentences. Finally, we have an output projection
```python
logits = layers.Dense(v_tgt, activation="softmax")(x)
transformer = Model([enc_inp, dec_inp], logits)
```
This gives us a probability distribution for each time step of possible next words in our translation.

Overall, this step was needed to build the Transformer architecture to translate each sentence by having the model first understand that order is important. Then we use the encoder block to process the sentence and understand the context, and then use the decoder block to generate the translation using the output from the encoder.

In this step, we see the code
```python
def smoothed_sparse_categorical_crossentropy(...):
class NoamSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
```
For crossentropy, we want to measure the model performance by measuring how well our model's predicted word matches the actual words so we can reduce loss and improve the translation quality and the BLEU score. For Noam scheduling, we want to increase the learning rate then decrease it because this would improve accuracy as well. Think of it as working out. If we stay on the heavy weights too long, it could hurt our performance in subsequent workout sessions. Similarly, if we keep the learning rate too high, it would hurt accuracy as the model continues to train. To maximize performance, we want to lower the rate as time goes on.

After compiling and training the model, we generate translations one token at a time by beam searching to find the most likely translations with alpha providing length normalizations to prevent the short sentences from dominating.
```python
def translate(sentence, beam_width=4, max_len=max_tgt_len, alpha=0.6):
```
Then we evaluate the BLEU score.
```python
result = bleu.compute(predictions=predictions, references=references)
```

From the table above, we can see that the highest BLEU score I attained was 13.97. Getting a score of 10 to 15 using a simple Transformer on a month-long project using one A100 GPU is a solid result. My model used an encoder-decoder architecture with six layers full and
```python
d_model = 512
```
which made it equivalent to the Transformer-base design. A large Transformer that would be using
```python
d_model = 1024.
```

However, I might be close to the performance plateau using only one GPU and using two million German-English sentence pairs. A PhD thesis from Stanford University in 2016 achieved a score of 20.7 that may have taken a few years of research and weeks of training [1]. Achieving a BLEU score of 20 or higher usually requires multiple GPUs or TPUs, larger datasets, and longer training durations. Because I was using only one A100 GPU, achieving such a high score would not be very realistic. In fact, achieving a higher BLEU score becomes increasingly difficult because that means we need more data to train on, longer training times, and more hardware. For example, in the 2017 paper "Attention Is All You Need", the authors used eight P100 GPUs, 4.5 million German-English sentence pairs, and 300,000 steps using between 25,000 to 32,000 tokens. Training took over 3 days, and they achieved a BLEU score of 27.3 [2]. While the P100 GPU is older than the A100 GPU, they used eight P100s so they had more hardware than I did and more teraflops at around 70 compared to 20 for me. In addition, translation tools such as DeepL and Google Translate achieved a score of at least 50, but that would require billions of parameters, hundreds to thousands of GPUs or TPUs, and weeks of training.

Lastly, human translations generally achieve a BLEU score of 60 to 70. No machine has achieve that high, but if it does, we should be wary because that is a good chance of overfitting [3]. As demonstrated above, our training BLEU was 89.80 with a test BLEU score being 0.20. We can see that it was overfitting so much that the model was essentially memorizing data.

Overall, this project demonstrates a successful implementation of a full Transformer trained from scratch using German-English sentence pairs. Despite hardware limitations, I was able to achieve a BLEU score of 13.97, a result within realistic range for an NMT academic project.

Sources:
1. https://nlp.stanford.edu/~manning/dissertations/thang-luong-thesis-augmented.pdf
2. https://arxiv.org/pdf/1706.03762v1
3. https://blog.modernmt.com/understanding-mt-quality-bleu-scores/#:~:text=Preferably%20using%20a%20team%20of%20translators%20who,small%20sample%20of%20250%20sentences%20are%20evaluated.